# 🎯 TestGap — Fine-Tuning do Modelo de IA no Google Colab (GPU Gratuita T4)

Este notebook executa o **Fine-Tuning Supervisionado (SFT)** com a técnica **QLoRA (4-bit)** em um modelo de código aberto (ex: **Qwen-2.5-Coder-7B-Instruct** ou **Llama-3-8B**) utilizando os dados gerados pela plataforma TestGap.

### ⚡ Pré-requisito no Colab:
Vá no menu superior: **Ambiente de execução (Runtime) -> Alterar tipo de ambiente de execução (Change runtime type)** e selecione **T4 GPU**.

## 1. Instalação das Bibliotecas de Machine Learning

In [ ]:
!pip install -q -U torch transformers datasets peft bitsandbytes accelerate trl

## 2. Enviar os arquivos do TestGap (`train.jsonl` e `val.jsonl`)
Execute a célula abaixo para fazer o upload dos arquivos que você exportou pelo comando:
`python scripts/export_dataset.py --mode splits` (localizados na pasta `data/exports/` da sua máquina).

In [ ]:
from google.colab import files
import os

print("Faça o upload de train.jsonl e val.jsonl:")
uploaded = files.upload()
print("Arquivos carregados:", list(uploaded.keys()))

## 3. Preparação dos Dados e Prompts de Instrução

In [ ]:
import json
from datasets import Dataset

def format_record(record):
    inp = record["input"]
    proj = record["project"]
    out = record["expected_output"]

    system_msg = (
        "Você é um engenheiro de software e auditor especialista em QA e lacunas de teste.\n"
        "Sua missão é analisar os requisitos, o código e a suíte de testes fornecida para "
        "identificar pontos cegos, casos de borda esquecidos e falhas de cobertura.\n"
        "Sua resposta deve ser estritamente em formato JSON válido, contendo as chaves 'has_gaps' e 'gaps'."
    )

    user_msg = (
        f"### Contexto do Projeto\n"
        f"Projeto: {proj['name']} (Linguagem: {proj['language']} | Framework: {proj['framework']})\n"
        f"Escopo da Análise: {inp['analysis_scope']}\n\n"
        f"### Requisitos de Software e Regras de Negócio:\n"
        f"{inp['requirements']}\n\n"
        f"### Código-Fonte da Aplicação:\n"
        f"{inp['source_code'] or 'Nenhum código fornecido (análise de caixa-preta).'}\n\n"
        f"### Suíte de Testes Existente:\n"
        f"{inp['test_suite']}\n\n"
        f"Analise a suíte de testes acima e determine se há lacunas de teste em relação aos requisitos."
    )

    assistant_msg = json.dumps(out, ensure_ascii=False, indent=2)
    return f"<|im_start|>system\n{system_msg}<|im_end|>\n<|im_start|>user\n{user_msg}<|im_end|>\n<|im_start|>assistant\n{assistant_msg}<|im_end|>"

def load_prompts(filename):
    with open(filename, 'r', encoding='utf-8') as f:
        return [format_record(json.loads(line)) for line in f if line.strip()]

train_texts = load_prompts("train.jsonl")
train_dataset = Dataset.from_dict({"text": train_texts})
print(f"Exemplos de treino preparados: {len(train_texts)}")

## 4. Carregar Modelo Base em 4-bit (QLoRA)

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

MODEL_ID = "Qwen/Qwen2.5-Coder-7B-Instruct"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True
)
model = prepare_model_for_kbit_training(model)

peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)
model = get_peft_model(model, peft_config)
model.print_trainable_parameters()

## 5. Executar o Treinamento (Fine-Tuning)

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./testgap-model-output",
    num_train_epochs=3,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    fp16=True,
    logging_steps=5,
    save_strategy="epoch",
    save_total_limit=1
)

trainer = SFTTrainer(
    model=model,
    train_dataset=train_dataset,
    dataset_text_field="text",
    max_seq_length=2048,
    tokenizer=tokenizer,
    args=training_args
)

trainer.train()
print("Treinamento finalizado com sucesso!")

## 6. Salvar e Baixar os Pesos Treinados (Adaptador LoRA)

In [ ]:
trainer.model.save_pretrained("testgap-lora-adapter")
tokenizer.save_pretrained("testgap-lora-adapter")

!zip -r testgap-lora-adapter.zip testgap-lora-adapter
files.download("testgap-lora-adapter.zip")
print("Download do adaptador concluído!")